# 09 — The product reads with both models, and refuses a changed file

**What this runs (GPU only, about 15–25 min on A100):** the 90 test images of the sealed mCellSeg test go through
the product's own pipeline (`culture.pipeline.analyze`), each read by the shipped model and by the approved
fine-tuned model `mcellseg_ftF_r2`. The fine-tuned reading is recorded beside the shipped one and decides nothing.
Then one image is read again with a copy of the fine-tuned file with one byte changed: the record must show the
refusal. Nothing is trained and nothing is scored against the experts.

**Before running:** upload `confluency_v5_bundle.zip` to `MyDrive/cultureqc/staged/`. The fine-tuned weights are
already there from nb/08 (`mcellseg_ftF_r2.weights`).

**I/O rule.** Nothing is read through the Drive mount except two single copies to local disk: the bundle and the
weights.

**Runtime:** A100 with **High-RAM** if offered.

**Bring back:** `MyDrive/cultureqc/staged/confluency_v5_results.zip` (under 1 MB).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time, subprocess, json, hashlib, zipfile
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
BUNDLE = f'{STAGE_DIR}/confluency_v5_bundle.zip'
WEIGHTS = f'{STAGE_DIR}/mcellseg_ftF_r2.weights'
RESULTS_ZIP = f'{STAGE_DIR}/confluency_v5_results.zip'
EXPECTED_COMMIT = 'f554898'        # the commit the bundle was made from
REPO = '/content/cultureqc'
assert os.path.exists(BUNDLE), f'upload the bundle first: {BUNDLE}'
assert os.path.exists(WEIGHTS), f'the fine-tuned weights from nb/08 are missing: {WEIGHTS}'
print(f'{BUNDLE}: {os.path.getsize(BUNDLE)/1e6:.1f} MB; weights {os.path.getsize(WEIGHTS)/1e9:.2f} GB')

In [ ]:
t0 = time.time()
shutil.rmtree(REPO, ignore_errors=True)
shutil.copy(BUNDLE, '/content/bundle.zip')                 # one sequential read from Drive
!unzip -q -o /content/bundle.zip -d $REPO
os.remove('/content/bundle.zip')
%cd $REPO
commit = open('COMMIT').read().strip()
print('bundle commit', commit, f'— unpacked in {time.time()-t0:.0f} s')
assert commit.startswith(EXPECTED_COMMIT), (commit, EXPECTED_COMMIT)

In [ ]:
!pip install -q cellpose==4.2.1.1 timm grad-cam tifffile scikit-image opencv-python-headless pyyaml transformers huggingface_hub jsonschema
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (A100)'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The shipped Cellpose-SAM weights Colab downloads must be the approved ones (configs/approved_changes.jsonl)
from cellpose import models
_ = models.CellposeModel(gpu=True)
w = os.path.join(str(models.MODEL_DIR), 'cpsam_v2')
h = hashlib.sha256(open(w, 'rb').read()).hexdigest()
print('cpsam_v2', h)
assert h == '0f1cc3f7ecdd8a037a57c6c48d9d8921391be4cbce3fa9f13c3e3a2e1253c667', 'not the shipped weights: stop and tell me'
# The fine-tuned weights: one copy from Drive to local disk, where the product looks for them
t0 = time.time()
dst = 'cache/finetune/mcellseg_ftF_r2/models/mcellseg_ftF_r2'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(WEIGHTS, dst)
h = hashlib.sha256(open(dst, 'rb').read()).hexdigest()
print('mcellseg_ftF_r2', h, f'— copied in {time.time()-t0:.0f} s')
assert h == '2d1549682c1da4829e7766bebf6e90e2373aa6c439f5138b098a71259d1239a2', 'not the approved fine-tuned weights: stop and tell me'
# The files that fix the rules are byte-identical to the commit
SEALED = {
 "scripts/finetuned_product_readings.py": "7e119e38640c2f7c018adde349738ea2beecbe2cac489b42f210db72355b3aac",
 "configs/approved_changes.jsonl": "455fcc1f65583155dc8831cacbf1936648a549a0bc56c78d1c5a97982b88e881",
 "configs/finetuned_models.yaml": "53e2687b0d65f6099dc5ef95e70a7b2013647c9f5fe82a9c2462e076f4bb9a52",
 "culture/approvals.py": "16710f519e51274758b28ec450cba8ef455743f717392cd765ff74b1cc169b3b",
 "culture/finetuned.py": "d99a4be893b835372b48c607d90bfa2b525c960b81f16d541dcc2ae925d90c70",
 "culture/pipeline.py": "3e8491526d9218c7fe7349345592a0f163aeac97273e7410428986f78045ef49",
 "culture/seg.py": "2d2440460de8c62e9440f54a6dc0f1edfe954c32d8b424c4f510b6a43b80db59",
 "results/confluency_mcellseg_split.json": "84f6fb7e4eb99bdefc5f07ce7aeace8f7d0bfb68ff8f4bd7422c8348ef029c15",
 "results/confluency_finetune_figure_curves.json": "75e454649670281f6c758efba5ac971aee4e7bb2715c8a44888611c3549a3968"
}
for f, s in SEALED.items():
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == s, f
print('committed files match', EXPECTED_COMMIT)

In [ ]:
# mCellSeg (Alam et al. 2026, CC BY 4.0) straight from Zenodo to local disk, MD5 from the record
t0 = time.time()
os.makedirs('/content/dl', exist_ok=True)
!curl -sL -o /content/dl/mCellSeg.zip "https://zenodo.org/api/records/20174259/files/mCellSeg.zip/content"
md5 = hashlib.md5(open('/content/dl/mCellSeg.zip', 'rb').read()).hexdigest()
print('md5', md5)
assert md5 == 'ed06d6e4c10e93b81703984cef852246'
os.makedirs('data/sources/mcellseg', exist_ok=True)
!unzip -q -o /content/dl/mCellSeg.zip -d data/sources/mcellseg
os.remove('/content/dl/mCellSeg.zip')
!ls data/sources/mcellseg/mCellSeg/labeled/images | wc -l
print(f'{time.time()-t0:.0f} s')

In [ ]:
# Masks only: the split must match the committed one
!python scripts/confluency_mcellseg.py count
# The product's own check, as it will run before each reading
!python scripts/approve_change.py check

### ⏸ Checkpoint
Above, you should see **`split matches the committed results/confluency_mcellseg_split.json`**, then
**`cpsam_v2: match`** and **`mcellseg_ftF_r2: match`**, and before that **`committed files match`**. If any of these
is missing, stop and send me the output.

In [ ]:
def save_results(label):
    # one small zip to Drive: the records + logs (no images, no weights)
    keep = ['results/finetuned_product_readings.jsonl']
    keep += [os.path.join('logs', f) for f in sorted(os.listdir('logs'))] if os.path.isdir('logs') else []
    local = '/content/confluency_v5_results.zip'
    with zipfile.ZipFile(local, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('COMMIT', commit)
        for p in keep:
            if os.path.exists(p):
                z.write(p)
    shutil.copy(local, RESULTS_ZIP)
    print(label, '->', RESULTS_ZIP, f'{os.path.getsize(local)/1e6:.2f} MB')
    with zipfile.ZipFile(local) as z:
        for n in z.namelist(): print('  ', n)

os.makedirs('logs', exist_ok=True)
def run(cmd, log):
    t0 = time.time()
    r = subprocess.run(['bash', '-c', f'set -o pipefail; {cmd} 2>&1 | tee -a logs/{log}.log | (grep -v -E "it/s]|s/it]" || true)'])
    print(f'[{log}] {(time.time()-t0)/60:.1f} min, exit {r.returncode}')
    if r.returncode != 0:
        save_results(f'{log} FAILED (partial results)')
        raise RuntimeError(f'{log} failed — send me logs/{log}.log (it is in the results zip)')

## Run — 90 product readings, then one with a changed byte

In [ ]:
run('python scripts/finetuned_product_readings.py run', 'product_readings')
save_results('Done')

## Done
Download `MyDrive/cultureqc/staged/confluency_v5_results.zip` and tell me where it is (Downloads is fine). If your
browser saves it as `confluency_v5_results(1).zip`, tell me that name.